In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")

In [2]:
con.execute("""
SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
       COUNT(*) AS orders,
       ROUND(SUM(item_revenue), 2) AS revenue,
       ROUND(SUM(item_revenue) / COUNT(*), 2) AS aov
FROM order_level
WHERE order_status = 'delivered'
  AND item_revenue IS NOT NULL
GROUP BY 1
ORDER BY 1
""").df()

,month,orders,revenue,aov
0,2016-09-01,1,134.97,134.97
1,2016-10-01,265,40325.11,152.17
2,2016-12-01,1,10.90,10.90
3,2017-01-01,750,111798.36,149.06
4,2017-02-01,1653,234223.40,141.70
5,2017-03-01,2546,359198.85,141.08
6,2017-04-01,2303,340669.68,147.92
7,2017-05-01,3546,489338.25,138.00
8,2017-06-01,3135,421923.37,134.58
9,2017-07-01,3872,481604.52,124.38


In [3]:
first_month = "2017-01-01"
last_month  = "2018-08-01"

con.execute(f"""
CREATE OR REPLACE TABLE monthly_summary AS
WITH monthly AS (
  SELECT DATE_TRUNC('month', order_purchase_timestamp) AS month,
         COUNT(*) AS orders,
         ROUND(SUM(item_revenue), 2) AS revenue,
         ROUND(SUM(item_revenue) / COUNT(*), 2) AS aov
  FROM order_level
  WHERE order_status = 'delivered'
    AND item_revenue IS NOT NULL
  GROUP BY 1
),
complete AS (
  SELECT * FROM monthly
  WHERE month BETWEEN DATE '{first_month}' AND DATE '{last_month}'
)
SELECT month, orders, revenue, aov,
       ROUND(100 * (revenue / LAG(revenue) OVER (ORDER BY month) - 1), 1) AS revenue_mom_pct,
       ROUND(100 * (orders  / LAG(orders)  OVER (ORDER BY month) - 1), 1) AS orders_mom_pct,
       ROUND(100 * (aov     / LAG(aov)     OVER (ORDER BY month) - 1), 1) AS aov_mom_pct
FROM complete
ORDER BY month
""")

con.execute("SELECT * FROM monthly_summary ORDER BY month").df()

,month,orders,revenue,aov,revenue_mom_pct,orders_mom_pct,aov_mom_pct
0,2017-01-01,750,111798.36,149.06,NaN,NaN,NaN
1,2017-02-01,1653,234223.40,141.70,109.5,120.4,-4.9
2,2017-03-01,2546,359198.85,141.08,53.4,54.0,-0.4
3,2017-04-01,2303,340669.68,147.92,-5.2,-9.5,4.8
4,2017-05-01,3546,489338.25,138.00,43.6,54.0,-6.7
5,2017-06-01,3135,421923.37,134.58,-13.8,-11.6,-2.5
6,2017-07-01,3872,481604.52,124.38,14.1,23.5,-7.6
7,2017-08-01,4193,554699.70,132.29,15.2,8.3,6.4
8,2017-09-01,4150,607399.67,146.36,9.5,-1.0,10.6
9,2017-10-01,4478,648247.65,144.76,6.7,7.9,-1.1


In [4]:
df = con.execute("SELECT * FROM monthly_summary ORDER BY month").df()

start, end = df.head(3), df.tail(3)

def change(col):
    return round(100 * (end[col].mean() / start[col].mean() - 1), 1)

print("Average of first 3 vs last 3 complete months")
print("Revenue change (%):", change("revenue"))
print("Orders change (%): ", change("orders"))
print("AOV change (%):    ", change("aov"))

Average of first 3 vs last 3 complete months
Revenue change (%): 263.4
Orders change (%):  276.0
AOV change (%):     -4.3


In [5]:
con.execute("COPY monthly_summary TO 'dashboard/monthly.csv' (HEADER, DELIMITER ',')")
print("saved")

saved


In [6]:
import pandas as pd

In [7]:
df = con.execute("SELECT * FROM monthly_summary ORDER BY month").df()
df["month"] = pd.to_datetime(df["month"])

In [8]:
first, last = df.iloc[0], df.iloc[-1]
start, end = df.head(3), df.tail(3)

In [9]:
def change(col):
    return round(100 * (end[col].mean() / start[col].mean() - 1), 1)

y17 = df[(df.month >= "2017-01-01") & (df.month <= "2017-08-01")]
y18 = df[(df.month >= "2018-01-01") & (df.month <= "2018-08-01")]
yoy_rev = round(100 * (y18.revenue.sum() / y17.revenue.sum() - 1), 1)
yoy_orders = round(100 * (y18.orders.sum() / y17.orders.sum() - 1), 1)
peak = df.loc[df.revenue.idxmax()]

In [10]:
print("=== COPY THIS INTO business_questions.md (Q1 answer) ===")
print(f"- Answer: Between {first.month:%b %Y} and {last.month:%b %Y}, monthly revenue grew from {first.revenue:,.0f} to {last.revenue:,.0f} and monthly orders from {int(first.orders):,} to {int(last.orders):,}. "
      f"Comparing the first 3 and last 3 complete months, orders rose {change('orders')}% while average order value changed {change('aov')}%, so growth came from more orders, not larger baskets. "
      f"Year-over-year (Jan-Aug 2018 vs Jan-Aug 2017), revenue rose {yoy_rev}% and orders rose {yoy_orders}%. "
      f"During Jan-Aug 2018 monthly revenue stayed between {y18.revenue.min():,.0f} and {y18.revenue.max():,.0f}, so growth flattened. "
      f"The peak month was {peak.month:%b %Y} ({peak.revenue:,.0f}), likely a seasonal effect such as Black Friday.")
print()
print("=== COPY THIS INTO README.md ===")
print(f"- Monthly analysis uses complete months only ({first.month:%b %Y} to {last.month:%b %Y}). Late 2016 and the final months of 2018 are excluded because volumes are too small or cut off.")

=== COPY THIS INTO business_questions.md (Q1 answer) ===
- Answer: Between Jan 2017 and Aug 2018, monthly revenue grew from 111,798 to 838,577 and monthly orders from 750 to 6,351. Comparing the first 3 and last 3 complete months, orders rose 276.0% while average order value changed -4.3%, so growth came from more orders, not larger baskets. Year-over-year (Jan-Aug 2018 vs Jan-Aug 2017), revenue rose 141.1% and orders rose 139.9%. During Jan-Aug 2018 monthly revenue stayed between 826,437 and 977,545, so growth flattened. The peak month was Nov 2017 (987,765), likely a seasonal effect such as Black Friday.

=== COPY THIS INTO README.md ===
- Monthly analysis uses complete months only (Jan 2017 to Aug 2018). Late 2016 and the final months of 2018 are excluded because volumes are too small or cut off.
